In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
import wandb
import os

In [6]:
# model_base_name = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"
base_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(base_model_name)

In [7]:
# Preparación del dataset autoregresivo
dataset_autoreg = load_dataset('text', data_files={'train': './data/sentences_train.txt'})
dataset_sft = load_dataset('json', data_files='./data/questions_train.jsonl', split="train")


def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=64)

tokenized_dataset_autoreg = dataset_autoreg.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_dataset_sft = dataset_sft.map(tokenize_function, batched=True, remove_columns=['text'])



In [8]:
# Configuración de LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, 
    r=8,  # rango de LoRA
    lora_alpha=32,  # hiperparámetro de LoRA
    lora_dropout=0.1,  # dropout de LoRA
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']  # módulos objetivo para aplicar LoRA
)

model = get_peft_model(model, lora_config)

# Configuración de entrenamiento
training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    save_steps=10_000,
    save_total_limit=2,
    logging_steps=100,
    learning_rate=1e-4,
    fp16=True,
    # evaluation_strategy="no",
    eval_steps=10_000,
    # report_to="wandb"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Configuración del Trainer
trainer_autoreg = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset_autoreg['train'],
    data_collator=data_collator
)

# Set supervised fine-tuning parameters
trainer_sft = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset_sft,
    # eval_dataset=valid_dataset_mapped,   
    # dataset_text_field="text",
    max_seq_length=64,
    tokenizer=tokenizer,
    args=training_args,
    # compute_metrics=compute_metrics_fn,
    packing=True,
    data_collator=data_collator,
)

c:\Users\NLP\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\utils\_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, packing. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
c:\Users\NLP\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(
c:\Users\NLP\AppData\Local\Programs\Python\Python312\Lib\site-packages\trl\trainer\sft_trainer.py:181: UserWarning: You passed a `packing` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
c:\Users\NLP\AppData\Local\Programs\Python\Python312\Lib\site-packages\trl\trainer\sft_trainer.py:269: Use

In [18]:
for epoch in range(10):
    print(f"Epoch {epoch+1}")
    print("Training autoregressive model...")
    trainer_autoreg.train()
    print("Training supervised fine-tuning model...")
    trainer_sft.train()

Epoch 1
Training autoregressive model...


Step,Training Loss
100,5.163400


Training supervised fine-tuning model...


Step,Training Loss


Epoch 2
Training autoregressive model...


Step,Training Loss
100,5.082200


... (17 outputs left out)

Step,Training Loss


Epoch 7
Training autoregressive model...


Step,Training Loss


KeyboardInterrupt: 

In [11]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Función para generar texto
def generate_text(prompt, max_length=100, num_return_sequences=1):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.01
    )
    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [25]:
# Ejemplo de uso
prompt = "<s>[INST] ¿Cuál es el teléfono de Federico? [/INST]"
generated_texts = generate_text(prompt, max_length=45, num_return_sequences=1)

# Imprimir los resultados generados
for i, text in enumerate(generated_texts):
    text_cleaned = text.split("</s>")[0]
    print(f"Generated Text {i+1}:\n{text_cleaned}\n")

Generated Text 1:
<s>[INST] ¿Cuál es el teléfono de Federico? [/INST] El teléfono de Federico es el 928204525. 

